# TP 5 — Projet de synthèse : carnet de notes

## Étape 1 — Données des étudiants

In [1]:
etudiants = {
    "ETU001": {"nom": "Miora", "notes": {"Python": [14, 16], "Maths": [12, 13]}},
    "ETU002": {"nom": "Tiana", "notes": {"Python": [11, 15], "Maths": [14, 10]}},
}
print("Effectif :", len(etudiants))

Effectif : 2


## Étape 2 — Décorateurs de journalisation et de validation

In [2]:
from functools import wraps
import json
from pathlib import Path

def journaliser(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        print("[JOURNAL]", f.__name__)
        return f(*args, **kwargs)
    return wrapper

def etudiant_existe(f):
    @wraps(f)
    def wrapper(matricule, *args, **kwargs):
        if matricule not in etudiants:
            raise KeyError("Matricule inconnu : " + matricule)
        return f(matricule, *args, **kwargs)
    return wrapper

## Étape 3 — Ajouter un étudiant et une note

In [3]:
@journaliser
def ajouter_etudiant(matricule, nom):
    if matricule in etudiants:
        print("Matricule déjà utilisé"); return False
    etudiants[matricule] = {"nom": nom, "notes": {}}
    return True

@journaliser
@etudiant_existe
def ajouter_note(matricule, matiere, note):
    if not isinstance(note, (int, float)) or not 0 <= note <= 20:
        raise ValueError("La note doit être entre 0 et 20")
    etudiants[matricule]["notes"].setdefault(matiere, []).append(note)
    return True

ajouter_etudiant("ETU003", "Hery")
ajouter_note("ETU003", "Python", 17)

[JOURNAL] ajouter_etudiant
[JOURNAL] ajouter_note


True

## Étape 4 — Moyennes par matière et générale

In [4]:
@etudiant_existe
def moyenne_matiere(matricule, matiere):
    notes = etudiants[matricule]["notes"].get(matiere, [])
    return sum(notes)/len(notes) if notes else None

@etudiant_existe
def moyenne_generale(matricule):
    notes = [n for liste in etudiants[matricule]["notes"].values() for n in liste]
    return sum(notes)/len(notes) if notes else None

print("Moyenne Python ETU001 :", moyenne_matiere("ETU001", "Python"))
print("Moyenne générale ETU001 :", moyenne_generale("ETU001"))

Moyenne Python ETU001 : 15.0
Moyenne générale ETU001 : 13.75


## Étape 5 — Classement et matières principales

In [5]:
def classement():
    resultats = []
    for matricule, info in etudiants.items():
        moyenne = moyenne_generale(matricule)
        if moyenne is not None:
            resultats.append((info["nom"], matricule, moyenne))
    return sorted(resultats, key=lambda x: x[2], reverse=True)

for rang, (nom, matricule, moyenne) in enumerate(classement(), 1):
    print(f"{rang}. {nom} ({matricule}) : {moyenne:.2f}/20")

def matieres_principales(matricule):
    notes = etudiants[matricule]["notes"]
    return sorted(notes, key=lambda m: sum(notes[m])/len(notes[m]), reverse=True)

print(matieres_principales("ETU001"))

1. Hery (ETU003) : 17.00/20
2. Miora (ETU001) : 13.75/20
3. Tiana (ETU002) : 12.50/20
['Python', 'Maths']


## Étape 6 — Sauvegarder et recharger en JSON

In [6]:
def sauvegarder(chemin="carnet_notes_ami.json"):
    Path(chemin).write_text(json.dumps(etudiants, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Sauvegardé dans :", Path(chemin).resolve())

def charger(chemin="carnet_notes_ami.json"):
    global etudiants
    etudiants = json.loads(Path(chemin).read_text(encoding="utf-8"))
    print("Étudiants chargés :", len(etudiants))

sauvegarder()

Sauvegardé dans : /home/cathysahy/jupyter/python/carnet_notes_ami.json


## Étape 7 — Menu interactif (facultatif)

In [7]:
def menu():
    while True:
        print("\n1 Classement | 2 Ajouter étudiant | 3 Ajouter note | 4 Sauvegarder | 0 Quitter")
        choix = input("Choix : ").strip()
        if choix == "1":
            for rang, (nom, matricule, moy) in enumerate(classement(), 1):
                print(rang, nom, matricule, f"{moy:.2f}/20")
        elif choix == "2":
            ajouter_etudiant(input("Matricule : ").strip(), input("Nom : ").strip())
        elif choix == "3":
            matricule = input("Matricule : ").strip()
            matiere = input("Matière : ").strip()
            try:
                ajouter_note(matricule, matiere, float(input("Note : ")))
            except (ValueError, KeyError) as e:
                print("Erreur :", e)
        elif choix == "4":
            sauvegarder()
        elif choix == "0":
            break
        else:
            print("Choix invalide")

# Décommente la ligne suivante pour démarrer le menu :
# menu()